In [1]:
import random, time
import numpy as np
import torch , torch.nn as nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms, models
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, classification_report

In [2]:
SEED = 42
BATCH_SIZE = 32
IMG_SIZE = 224
VAL_FRAC = 0.2
EPOCHS_HEAD, EPOCHS_FT = 3, 7
LR_HEAD, LR_FT = 1e-3, 1e-4

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
print(device)

cpu


In [3]:
mean, std = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]  # ImageNet stats

train_tf = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.7, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(0.2, 0.2, 0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean, std),
])
eval_tf = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean, std),
])

In [5]:
full_train_aug  = datasets.OxfordIIITPet("data", split="trainval", target_types="category", transform=train_tf, download=True)
full_train_eval = datasets.OxfordIIITPet("data", split="trainval", target_types="category", transform=eval_tf)
test_ds         = datasets.OxfordIIITPet("data", split="test",     target_types="category", transform=eval_tf)

class_names = full_train_aug.classes
n = len(full_train_aug)
idx = torch.randperm(n, generator=torch.Generator().manual_seed(SEED)).tolist()
n_val = int(n * VAL_FRAC)
val_idx, train_idx = idx[:n_val], idx[n_val:]

train_ds = Subset(full_train_aug, train_idx)   # augmented
val_ds   = Subset(full_train_eval, val_idx)    # not augmented

print(len(train_ds), len(val_ds), len(test_ds), len(class_names))

2944 736 3669 37
